<a href="https://www.kaggle.com/code/emmanuelniyioriolowo/4-classical-models?scriptVersionId=284125627" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Imports

In [81]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/baseline-models/metrics_table.csv
/kaggle/input/baseline-models/__results__.html
/kaggle/input/baseline-models/__notebook__.ipynb
/kaggle/input/baseline-models/__output__.json
/kaggle/input/baseline-models/custom.css
/kaggle/input/ncdc-lassa-fever-timeseries-20202025/lassa_fever_timeseries_minimal.csv
/kaggle/input/ncdc-lassa-fever-timeseries-20202025/lassa_fever_timeseries_full.csv


# Functions

In [82]:
# Mean Absolute Error (MAE)
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

# Root Mean Squared Error (RMSE)
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

# Mean Absolute Percentage Error (MAPE)
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0  # avoid division by zero
    return (np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

# Function to calculate all three metrics
def calculate_evaluation_metrics(y_true, y_pred):
    metrics = []
    metrics.append(mae(y_true, y_pred).round(2))
    metrics.append(rmse(y_true, y_pred).round(2))
    metrics.append(mape(y_true, y_pred).round(2))
    return metrics


# plot graph 
def plot_forecasts(forecast, title):
    """Function to plot the forecasts"""
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=train.index, y=train['confirmed_cases'], name="Train", line=dict(color='#1f77b4')))
    fig.add_trace(go.Scatter(x=test.index, y=test['confirmed_cases'], name="Test", line=dict(color='#2ca02c')))
    fig.add_trace(go.Scatter(x=test.index, y=forecast, name="Forecast", line=dict(color='#ff0000')))
    fig.update_layout(template="simple_white", font=dict(size=18), title_text=title,
                     width=650, title_x=0.5, height=400, xaxis_title='Date',
                     yaxis_title='Confirmed Cases')
    return fig.show()


metrics_table = pd.read_csv('/kaggle/input/baseline-models/metrics_table.csv')

# Data Extraction and EDA

In [83]:
# read data into a dataframe
df = pd.read_csv('/kaggle/input/ncdc-lassa-fever-timeseries-20202025/lassa_fever_timeseries_minimal.csv')

# construct series dataframe
ts_data = df[['week_end_date', 'epi_week', 'confirmed_cases']].copy()
ts_data.columns = ['date', 'epi_week', 'confirmed_cases']

# Ensure datetime index
ts_data['date'] = pd.to_datetime(ts_data['date'])
ts_data = ts_data.set_index('date')
ts_data = ts_data.asfreq('W') # set frequency to weekly
ts_data

,epi_week,confirmed_cases
date,,
2020-01-05,1,18
2020-01-12,2,64
2020-01-19,3,81
2020-01-26,4,95
2020-02-02,5,104
...,...,...
2025-10-19,42,9
2025-10-26,43,11
2025-11-02,44,12


In [84]:
# Some basic EDA
print(ts_data.dtypes)
print(ts_data.isnull().sum())
print(ts_data.describe())

epi_week           int64
confirmed_cases    int64
dtype: object
epi_week           0
confirmed_cases    0
dtype: int64
         epi_week  confirmed_cases
count  307.000000       307.000000
mean    26.136808        20.540717
std     14.879500        25.897753
min      1.000000         0.000000
25%     13.000000         6.000000
50%     26.000000        10.000000
75%     39.000000        21.000000
max     53.000000       137.000000


In [85]:
# create and display plot 
fig =  px.line(ts_data, x=ts_data.index, y="confirmed_cases",
        title="Weekly Lassa Fever Cases (2020–2025)")
fig.show()

# Train Test Split

In [86]:
# Train Test Split
train = ts_data.iloc[:-int(len(ts_data)*0.2)].copy()
test = ts_data.iloc[-int(len(ts_data)*0.2):].copy()

# SARIMA

## 

In [87]:
# seasonal period for weekly data with yearly seasonality:
s = 52

In [88]:
# make output directories 
os.makedirs("outputs/plots", exist_ok=True)
os.makedirs("outputs/models", exist_ok=True)

In [89]:
# Import packages
from scipy.special import inv_boxcox
from statsmodels.tsa.arima.model import ARIMA

# Build ARIMA model
model = ARIMA(train['confirmed_cases'], order=(1, 1, 0),
              seasonal_order=(0, 1, 1, 52)).fit()
test['sarima_forecasts'] = model.forecast(len(test))

# Plot the forecasts
plot_forecasts(sarima_forecasts, 'SARIMA Forecast')

# Calculate metrics and add to results dataframe
metrics_list = ['SARIMA'] + calculate_evaluation_metrics(test['confirmed_cases'], test['sarima_forecasts'])
metrics_table.loc[len(metrics_table)] = metrics_list

/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/statespace/sarimax.py:1009: UserWarning:

Non-invertible starting seasonal moving average Using zeros as starting parameters.



In [90]:
metrics_table

,Model,MAE,RMSE,MAPE
0,Naive,16.33,26.37,60.30
1,Average,16.20,20.94,129.03
2,Drift,17.69,27.12,69.84
3,Seasonal Naive,8.74,15.53,49.24
4,Seasonal Mean,7.91,12.25,44.11
5,SARIMA,7.70,12.50,42.58


In [91]:
# import pmdarima as pm
# from statsmodels.tsa.stattools import acf, pacf
# from statsmodels.stats.diagnostic import acorr_ljungbox

# # SARIMA (auto_arima to suggest orders)
# print("Running auto_arima to identify candidate orders. This may take a minute...")
# auto = pm.auto_arima(train,
#                      seasonal=True,
#                      m=s,
#                      stepwise=True,
#                      error_action='ignore',
#                      suppress_warnings=True,
#                      trace=True,
#                      max_p=5, max_q=5, max_P=2, max_Q=2,
#                      information_criterion='aicc')  # aicc helps small samples

# print("auto_arima found:", auto.summary())

# # Recommended orders
# order = auto.order          # (p,d,q)
# seasonal_order = auto.seasonal_order  # (P,D,Q,s)